In [141]:
import pandas as pd

# ---- Claims ----
df_claims = {
    "fever":       pd.read_csv("_claim_datasets/fever_1k.csv"),
    "fever-fixed": pd.read_csv("_claim_datasets/fever_1k_fixed.csv"),
    "scifact":     pd.read_csv("_claim_datasets/scifact_693.csv"),
    "averitec":    pd.read_csv("_claim_datasets/averitec_3017.csv"),
}

# ---- Results ----
def read_jsonl(path):
    return pd.read_json(path, lines=True)

df_results = {
    "fever": {
        "gpt-4o":       read_jsonl("_results/fever_1k/gpt_results_raw_gpt-4o.jsonl"),
        "gpt-5":        read_jsonl("_results/fever_1k/gpt_results_raw_gpt-5.jsonl"),
        "originality":  read_jsonl("_results/fever_1k/checker_results.jsonl"),
    },
    "scifact": {
        "gpt-4o":       read_jsonl("_results/scifact_693/gpt_results_raw_gpt-4o.jsonl"),
        "gpt-5":        read_jsonl("_results/scifact_693/gpt_results_raw_gpt-5.jsonl"),
        "originality":  read_jsonl("_results/scifact_693/scifact_checker_results.jsonl"),
    },
    "averitec": {
        "gpt-4o":       read_jsonl("_results/averitec_3017/gpt_results_raw_gpt-4o.jsonl"),
        "gpt-5":        read_jsonl("_results/averitec_3017/gpt_results_raw_gpt-5.jsonl"),
        "originality":  read_jsonl("_results/averitec_3017/averitec_checker_results.jsonl"),
    },
}

for dataset in ["fever", "scifact", "averitec"]:
    for model in ["originality", "gpt-4o", "gpt-5"]:
        jsonl = df_results[dataset][model]
        jsonl["id"] = jsonl[f"{dataset}_id"]
        if model == "originality":
            jsonl["explanation"] = jsonl["checker_response"].apply(
                lambda x: x["data"]["results"][0]["explanation"] if x["data"]["results"] else '-'
            )
            jsonl["label"] = jsonl["checker_response"].apply(
                lambda x: x["data"]["results"][0]["classification"] == 'True' if x["data"]["results"] else '-'
            )
        else:
            jsonl["explanation"] = jsonl["response_text"]
            # "The claim is FALSE"
            # "The claim is TRUE"
            jsonl["label"] = jsonl["response_text"].apply(
                lambda x: (True if (x.startswith('TRUE') or x.startswith('The claim is TRUE')) else (False if (x.startswith('FALSE') or x.startswith('The claim is FALSE')) else '-')) if isinstance(x, str) else '-'
            )
        df = jsonl[["id", "claim", "label", "explanation"]]
        df_results[dataset][model] = df
